In [15]:
import os
import json
from dataclasses import dataclass
import time

from mistralai import Mistral, ChatCompletionResponse


MISTRAL_API_KEY=os.getenv("MISTRAL_API_KEY")
MISTRAL_MODEL = "mistral-small-latest"

In [16]:
@dataclass
class WordDataPoint:
    def __init__(self, word: str, definition: str, example: str):
        self.word = word
        self.definition = definition
        self.example = example

    def __repr__(self):
        return f"WordDataPoint(word={self.word}, definition={self.definition}, example={self.example})"
    
    def to_dict(self):
        return {
            "word": self.word,
            "definition": self.definition,
            "example": self.example
        }
    
class CustomMistralClient:
    def __init__(self):
        self.api_key = MISTRAL_API_KEY
        self.client = Mistral(api_key=self.api_key)

    def get_word_example(self, word: str) -> dict:
        system_message = "You are an expert English language professional. Provide a concise example sentence using the given word."
        user_prompt = f"Provide a concise example sentence using the word: '{word}'."
        response : ChatCompletionResponse = self.client.chat.complete(
            model=MISTRAL_MODEL,
            messages=[
                {"role":"system", "content": system_message},
                {"role": "user", "content": user_prompt}
            ]
        )
        time.sleep(1)  # Wait for 1 second to handle rate limiting
        return response.choices[0].message.content
    
    def get_word_definition(self, word:str)->str:
        system_message = "You are an expert English language professional. Provide a concise definition of the given word."
        user_prompt = f"Provide a concise definition of the word: '{word}'."
        response : ChatCompletionResponse = self.client.chat.complete(
            model=MISTRAL_MODEL,
            messages=[
                {"role":"system", "content": system_message},
                {"role": "user", "content": user_prompt}
            ]
        )
        time.sleep(1)  # Wait for 1 second to handle rate limiting
        return response.choices[0].message.content


def load_file(file_path) -> list[str]:
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.readlines()
    return content

def output_file(file_name:str, data: list[WordDataPoint])->None:
    output_directory_path = "cleaned_data"
    if(not os.path.exists(output_directory_path)):
        os.mkdir(output_directory_path)
    file_path = os.path.join(output_directory_path,file_name)
    with open(file_path,"w",encoding='utf-8') as file:
        json.dump([data_point.to_dict() for data_point in data],file,indent=4)

In [17]:
directory_path = "./corpus"
files = os.listdir(directory_path)
files

['definitions_1.txt']

In [18]:
content = load_file(os.path.join(directory_path, files[0]))
content[:3]

['a feeling of\xa0pensive sadness, typically with no obvious cause;;melancholy --> an air of melancholy surrounded him\n',
 '\n',
 'engaged in, involving, or reflecting deep or serious thought;; pensive --> a pensive mood\n']

In [19]:
# clean the data and create a list of words instead
# use the list of words to call an API to get definitions and examples 
data_points : list[WordDataPoint] = []
api_agent = CustomMistralClient()
for line in content: 
    data_point = None
    if ";;" in line and "-->" in line:
        parts = line.split(";;")
        definition = parts[0].strip()
        word_example = parts[1].split("-->")
        word = word_example[0].strip()
        example = word_example[1].strip()
        data_point = WordDataPoint(word=word, definition=definition, example=example)
    elif ";;" in line:
        parts = line.split(";;")
        definition = parts[0].strip()
        word = parts[1].strip()
        example = api_agent.get_word_example(word)
        data_point = WordDataPoint(word=word, definition=definition, example=example)
    else:
        word = line.strip()
        if word:
            definition = api_agent.get_word_definition(word)
            example = api_agent.get_word_example(word)
            data_point = WordDataPoint(word=word, definition=definition, example=example) 
    if data_point:
        data_points.append(data_point)

In [20]:
data_points[:3]

[WordDataPoint(word=melancholy, definition=a feeling of pensive sadness, typically with no obvious cause, example=an air of melancholy surrounded him),
 WordDataPoint(word=pensive, definition=engaged in, involving, or reflecting deep or serious thought, example=a pensive mood),
 WordDataPoint(word=polemic, definition=a speech or piece of writing expressing a strongly critical attack on or controversial opinion about someone or something, example=his polemic against that article)]

In [21]:
output_file(files[0], data_points)